# Exploratory Data Analysis (EDA)

## Importing Libraries

In [29]:
import pandas as pd
from pathlib import Path
from data_profiling import ProfileReport

## Load CSV Files

In [30]:
csv_folder = Path("./movies-database")

movies       = pd.read_csv(csv_folder / "movies.csv")
links        = pd.read_csv(csv_folder / "links.csv")
ratings      = pd.read_csv(csv_folder / "ratings.csv")
tags         = pd.read_csv(csv_folder / "tags.csv")
genome_scores = pd.read_csv(csv_folder / "genome-scores.csv")
genome_tags  = pd.read_csv(csv_folder / "genome-tags.csv")

## Merge DataFrames

Each file shares a key with at least one other file:

| File | Key(s) |
|---|---|
| `movies` | `movieId` |
| `links` | `movieId` |
| `ratings` | `movieId`, `userId` |
| `tags` | `movieId`, `userId` |
| `genome_scores` | `movieId`, `tagId` |
| `genome_tags` | `tagId` |

We build two separate merged DataFrames depending on the analysis goal:
- **`df_ratings`**: user-level rating data with movie metadata
- **`df_genome`**: genome relevance scores with tag labels and movie titles


- timestamp_rating: renamed from the original timestamp column in ratings.csv before the merge. It records when a user submitted their numeric star rating for a movie. Every rating row has this value (no nulls).
- timestamp_tag:  created during the tags_agg groupby as ("timestamp", "max"). It records the most recent moment a user applied any tag to that movie. Because most rating rows have no corresponding tag at all, this column is sparse.
- The rename of timestamp → timestamp_rating is necessary because the tags aggregation is about to introduce timestamp_tag into the same DataFrame.

1. Pre-aggregate the tags table. Since a single user can attach multiple tags to the same movie, a direct merge would fan out the ratings rows (one row per tag). To avoid this, the tags table is first collapsed by (userId, movieId): all tag strings are concatenated into a single pipe-separated value (e.g. "funny|classic")
2. Chain three left merges on ratings as the base: All merges are how="left", so every rating row is preserved. Rows without a matching tag simply get NaN for the tag columns.

In [31]:
# User-level view: ratings with movie info, links, and user tags

tags_agg = (
    tags
    .groupby(["userId", "movieId"], as_index=False)
    .agg(
        tag=("tag", lambda s: "|".join(s.dropna())),
        timestamp_tag=("timestamp", "max")
    )
)

df_ratings = (
    ratings.rename(columns={"timestamp": "timestamp_rating"})
    .merge(movies, on="movieId", how="left")
    .merge(links,  on="movieId", how="left")
    .merge(tags_agg, on=["userId", "movieId"], how="left")
)

print(df_ratings.shape)
df_ratings.head()

(25000095, 10)


,userId,movieId,rating,timestamp_rating,title,genres,imdbId,tmdbId,tag,timestamp_tag
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,110912,680.0,NaN,NaN
1,1,306,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,111495,110.0,NaN,NaN
2,1,307,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,108394,108.0,NaN,NaN
3,1,665,5.0,1147878820,Underground (1995),Comedy|Drama|War,114787,11902.0,NaN,NaN
4,1,899,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance,45152,872.0,NaN,NaN


In [32]:
# Genome view: relevance scores with tag names and movie titles
df_genome = (
    genome_scores
    .merge(genome_tags, on="tagId",   how="left")   # add tag label
    .merge(movies,      on="movieId", how="left")   # add title & genres
)

print(df_genome.shape)
df_genome.head()

(15584448, 6)


,movieId,tagId,relevance,tag,title,genres
0,1,1,0.02875,007,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,2,0.02375,007 (series),Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,1,3,0.06250,18th century,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,1,4,0.07575,1920s,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,1,5,0.14075,1930s,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


## Generate Profile Report

Run a profile report on the ratings-based DataFrame (most analytical value).

In [33]:
profile = ProfileReport(df_ratings, title="Movie Ratings EDA", explorative=True)
profile.to_file("eda_ratings_report.html")
print("Report saved to eda_ratings_report.html")
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
profile = ProfileReport(df_genome, title="Movie Genomes EDA", explorative=True)
profile.to_file("eda_genomes_report.html")
print("Report saved to eda_genomes_report.html")
profile